<a href="https://colab.research.google.com/github/Lauralug0/GBC/blob/main/notebooks/news_topic_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# News Modeling

Topic modeling involves **extracting features from document terms** and using
mathematical structures and frameworks like matrix factorization and SVD to generate **clusters or groups of terms** that are distinguishable from each other and these clusters of words form topics or concepts

Topic modeling is a method for **unsupervised classification** of documents, similar to clustering on numeric data

These concepts can be used to interpret the main **themes** of a corpus and also make **semantic connections among words that co-occur together** frequently in various documents

Topic modeling can help in the following areas:
- discovering the **hidden themes** in the collection
- **classifying** the documents into the discovered themes
- using the classification to **organize/summarize/search** the documents

Frameworks and algorithms to build topic models:
- Latent semantic indexing
- Latent Dirichlet allocation
- Non-negative matrix factorization

## Latent Dirichlet Allocation (LDA)
The latent Dirichlet allocation (LDA) technique is a **generative probabilistic model** where each **document is assumed to have a combination of topics** similar to a probabilistic latent semantic indexing model

In simple words, the idea behind LDA is that of two folds:
- each **document** can be described by a **distribution of topics**
- each **topic** can be described by a **distribution of words**

### LDA Algorithm

- 1. For each document, **randomly initialize each word to one of the K topics** (k is chosen beforehand)
- 2. For each document D, go through each word w and compute:
    - **P(T |D)** , which is a proportion of words in D assigned to topic T
    - **P(W |T )** , which is a proportion of assignments to topic T over all documents having the word W
- **Reassign word W with topic T** with probability P(T |D)´ P(W |T ) considering all other words and their topic assignments

![LDA](https://raw.githubusercontent.com/subashgandyer/datasets/main/images/LDA.png)

### Steps
- Install the necessary library
- Import the necessary libraries
- Download the dataset
- Load the dataset
- Pre-process the dataset
    - Stop words removal
    - Email removal
    - Non-alphabetic words removal
    - Tokenize
    - Lowercase
    - BiGrams & TriGrams
    - Lemmatization
- Create a dictionary for the document
- Filter low frequency words
- Create an Index to word dictionary
- Train the Topic Model
- Predict on the dataset
- Evaluate the Topic Model
    - Model Perplexity
    - Topic Coherence
- Visualize the topics

### Install the necessary library

In [ ]:
!pip install pyLDAvis gensim spacy

### Import the libraries

In [ ]:
import json
import re
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
import spacy
import nltk
from nltk.corpus import stopwords
import matplotlib.pyplot as plt

### Download the dataset
Dataset: https://raw.githubusercontent.com/subashgandyer/datasets/main/newsgroups.json

#### 20-Newsgroups dataset
- 11K newsgroups posts
- 20 news topics

### Load the dataset

In [ ]:
with open('newsgroups.json', 'r') as f:
    data_loaded = json.load(f)

data = [re.sub(r'\n', '', entry_text) for entry_text in data_loaded['content'].values()]

In [ ]:
print(data[:1])

### Preprocess the data

### Email Removal

In [ ]:
data_no_emails = [re.sub(r'\S*@\S*\s?', '', entry_text) for entry_text in data]

In [ ]:
print(data_no_emails[:1])

### Newline Removal

In [ ]:
data_no_newlines = [re.sub('\s+', ' ', entry_text) for entry_text in data_no_emails]
print(data_no_newlines[:1])

### Single Quotes Removal

In [ ]:
data_no_quotes = [re.sub("'", '', entry_text) for entry_text in data_no_newlines]

### Tokenize
- Create **sent_to_words()**
    - Use **gensim.utils.simple_preprocess**
    - Use **generator** instead of an usual function

In [ ]:
def sent_to_words(sentences):
    for sentence in sentences:
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))

In [ ]:
data_words = list(sent_to_words(data_no_quotes))
print(data_words[:1])

### Stop words Removal
- Extend the stop words corpus with the following words
    - from
    - subject
    - re
    - edu
    - use

In [ ]:
nltk.download('stopwords')
stop_words = stopwords.words('english')
stop_words.extend(['from', 'subject', 're', 'edu', 'use'])


#### remove_stopwords( )

In [ ]:
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]

In [ ]:
data_words_no_stopwords = remove_stopwords(data_words)
print(data_words_no_stopwords[:1])

### Bigrams
- Use **gensim.models.Phrases**
- 100 as threshold

In [ ]:

bigram = gensim.models.Phrases(data_words_no_stopwords, min_count=5, threshold=100)

In [ ]:

bigram_mod = gensim.models.phrases.Phraser(bigram)

#### make_bigrams( )

In [ ]:
def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

In [ ]:
data_words_bigrams = make_bigrams(data_words_no_stopwords)

### Lemmatization
- Use spacy
    - Download spacy en model (if you have not done that before)
    - Load the spacy model

In [ ]:
!python -m spacy download en

In [ ]:
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

#### lemmatizaton( )

In [ ]:
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent))
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [ ]:
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

In [ ]:
print(data_lemmatized[:1])

### Create a Dictionary

In [ ]:
id2word = corpora.Dictionary(data_lemmatized)

### Create Corpus

In [ ]:
corpus = [id2word.doc2bow(text) for text in data_lemmatized]

### Filter low-frequency words

In [ ]:

id2word.filter_extremes(no_below=15, no_above=0.5, keep_n=100000)

corpus = [id2word.doc2bow(text) for text in data_lemmatized]

print(f"Number of unique tokens after filtering: {len(id2word)}")
print(f"First 10 entries of the filtered corpus: {corpus[:10]}")

### Create Index 2 word dictionary

In [ ]:

print('Sample of (id, word) mapping:')
for i in range(min(5, len(id2word))):
    print(f'  {i}: {id2word[i]}')

In [ ]:
print(id2word.token2id)
print(id2word)

### Build a News Topic Model

#### LdaModel
- **num_topics** : this is the number of topics you need to define beforehand
- **chunksize** : the number of documents to be used in each training chunk
- **alpha** : this is the hyperparameters that affect the sparsity of the topics
- **passess** : total number of training assess

In [ ]:
lda_model = gensim.models.LdaMulticore(
    corpus=corpus,
    id2word=id2word,
    num_topics=20,
    random_state=100,
    chunksize=2000,
    passes=10,
    alpha='asymmetric',
    per_word_topics=True
)

### Print the Keyword in the 10 topics

In [ ]:
print(lda_model.print_topics(num_words=10))

## Evaluation of Topic Models
- Model Perplexity
- Topic Coherence

### Model Perplexity

Model perplexity is a measurement of **how well** a **probability distribution** or probability model **predicts a sample**

In [ ]:

print('Perplexity: ', lda_model.log_perplexity(corpus))

### Topic Coherence
Topic Coherence measures score a single topic by measuring the **degree of semantic similarity** between **high scoring words** in the topic.

In [ ]:

coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('Coherence Score: ', coherence_lda)

### Visualize the Topic Model
- Use **pyLDAvis**
    - designed to help users **interpret the topics** in a topic model that has been fit to a corpus of text data
    - extracts information from a fitted LDA topic model to inform an interactive web-based visualization

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models as gensim_models

pyLDAvis.enable_notebook()
vis = gensim_models.prepare(lda_model, corpus, id2word)
vis